In [1]:
import numpy as np
import pandas as pd
from scipy import stats
import pingouin as pg

# ----------------------------
# 1) Raw LH frequecny data 
# ----------------------------
data5D = {
    "ConFoff": {
        "Before": [18.3, 12.0, 15.0, 15.0, 17.5, 15.0],
        "After":  [22.5, 22.5, 27.5, 20.0, 25.0, 25.0],
    },
    "No stimulation": {
        "Before": [16.67, 15.0, 16.67, 15.0, 16.67, 15.0],
        "After":  [16.77, 18.33, 15.0, 20.0, 15.0, 15.0],
    },
    "Control virus": {
        "Before": [22.5, 20.0, 15.0, 18.33, 20.0, 18.33],
        "After":  [18.33, 20.0, 15.0, 15.0, 18.33, 20.0],
    }
}

data5J = {
    "ConFoff": {
        "Before": [20.00, 16.67, 18.33, 15.00, 17.50, 20.00],
        "After":  [13.75, 16.67, 15.00, 20.00, 17.50, 16.70],
    },
    "No stimulation": {
        "Before": [15.00, 11.25, 17.50, 20.00, 16.67],
        "After":  [17.5, 12.5, 15.0, 17.5, 16.67],
    }
}

data6D = {
    "ConFoff": {
        "Before": [22.5, 15, 12.5, 22.5, 20],
        "After":  [32.5, 30, 60, 50, 75],
    },
    "No stimulation": {
        "Before": [22.5, 15, 35, 20, 13.75],
        "After":  [20, 22.5, 15, 22.5, 16.67],
    },
    "Control virus": {
        "Before": [16.67, 17.5, 15, 15, 18.33],
        "After":  [18.33, 20, 18.33, 16.67, 16.67],
    }
}

data6J = {
    "ConFoff": {
        "Before": [15, 17.5, 16.66666667, 22.5, 15, 13.33],
        "After":  [15, 16.66666667, 15, 20, 20, 25],
    },
    "No stimulation": {
        "Before": [13.75, 17.5, 20, 13.75, 16.67],
        "After":  [17.5, 17.5, 15, 16.67, 20],
    }
}





In [3]:
data = data5D

# ----------------------------
# 2) Convert to dataframe
# ----------------------------
rows = []
subj_id = 0

for condition, times in data.items():
    for i in range(times["Before"].__len__()):  
        subj_id += 1
        rows.append({"Subject": subj_id, "Condition": condition,
                     "Time": "Before", "Value": times["Before"][i]})
        rows.append({"Subject": subj_id, "Condition": condition,
                     "Time": "After", "Value": times["After"][i]})

df = pd.DataFrame(rows)

# ----------------------------
# 3) Mixed ANOVA
# ----------------------------
anova = pg.mixed_anova(
    data=df,
    dv="Value",
    within="Time",
    between="Condition",
    subject="Subject"
)

for _, r in anova.iterrows():
    aa = float(r["p-unc"])

print("\n=== Mixed ANOVA (Time × Condition) ===")
print(f"{anova.Source[2]}: F({anova.DF1[2]}, {anova.DF2[2]}) = {anova.F[2]:.2f}, p = {aa:.6f}, partial η² = {anova.np2[2]:.2f}")
#print(anova)

# ----------------------------
# 4) Paired post-hoc tests (Before vs After within each condition)
# ----------------------------
print("\n=== Paired Before vs After comparisons ===")

for condition in df["Condition"].unique():
    sub = df[df["Condition"] == condition]
    before = sub[sub["Time"] == "Before"]["Value"].values
    after  = sub[sub["Time"] == "After"]["Value"].values

    # Paired t-test
    t, p = stats.ttest_rel(after, before)
    df_t = len(before) - 1

    # Mean difference
    mean_diff = after.mean() - before.mean()

    # Cohen's dz (paired effect size)
    dz = mean_diff / np.std(after - before, ddof=1)

    # 95% CI for mean difference
    diff = after - before
    se = stats.sem(diff)
    tcrit = stats.t.ppf(0.975, df_t)
    ci_low = mean_diff - tcrit * se
    ci_high = mean_diff + tcrit * se

    print(
        f"{condition}: mean difference (After−Before) = {mean_diff:.2f}; "
        f"t({df_t}) = {t:.2f}, p = {p:.6f}, "
        f"Cohen’s dz = {dz:.2f}, "
        f"95% CI = [{ci_low:.2f}, {ci_high:.2f}]"
    )



=== Mixed ANOVA (Time × Condition) ===
Interaction: F(2, 15) = 19.51, p = 0.000067, partial η² = 0.72

=== Paired Before vs After comparisons ===
ConFoff: mean difference (After−Before) = 8.28; t(5) = 6.19, p = 0.001604, Cohen’s dz = 2.53, 95% CI = [4.84, 11.72]
No stimulation: mean difference (After−Before) = 0.85; t(5) = 0.76, p = 0.481420, Cohen’s dz = 0.31, 95% CI = [-2.02, 3.72]
Control virus: mean difference (After−Before) = -1.25; t(5) = -1.38, p = 0.226639, Cohen’s dz = -0.56, 95% CI = [-3.58, 1.08]
